# Notebook 04 — Stress Testing & Fairness

NB04 is the most claim-authorization-sensitive notebook in the Fannie Mae pipeline. It computes the scenario ECLs, capital coverage diagnostics, severity check, and fairness results — but it **never writes a verdict**. Every section that could produce a "PASS"/"FAIL"/"Adequate"/"Inadequate" token has been reframed to produce a neutral label or a computed number instead.

The two areas the reader must understand:

- **"Capital coverage" replaces "capital adequacy."** The notebook can answer "how much of an assumed asset base would be consumed by ECL under stress?" It cannot answer "does the bank have enough capital?" The ECL/CET1 ratio is computed and explicitly labelled "DIAGNOSTIC (NOT capital adequacy)."

- **"Scenario severity check" replaces "scenario validation."** The 2008 comparison is directional (is the scenario at least as severe as 2008?), not empirical (does the scenario reproduce 2008?).

The notebook's claim-authorization assertion at the end of Section 8 ('assert 'PASS' not in str(r)...') and its C5 scope guard ('assert 'roc_curve' not in dir()') are the enforcement.

### Interpreting the key results

**Scenario ECLs (per loan): baseline $835, adverse $1,803, severely adverse $3,756, tail $7,512**. These come from NB03's contract values for PD/LGD/EAD, multiplied by the scenario multipliers and weighted. The reader should understand:

- **The scenario ECLs are per-loan, not portfolio**. To get portfolio figures, multiply by portfolio size (100,000 loans). The notebook does this in the capital coverage section.

- **The PD cap does not fire at the per-loan scenario level** (tail PD = 4.29%, well below the 20% cap). The cap effect is 0.0% understated. This is the correct result but a reader should understand it is run-specific — if base PD were higher, the cap would fire and the tail ECL would be understated. The narrative should state the cap-effect figure and the conditions under which it would be non-zero.

**Capital coverage: ECL as % of total assets (weighted) is 22.31%; tail is 75.12%. ECL/CET1 is 495.8% (weighted) and 1669.4% (tail)**. The reader must understand why the ECL/CET1 ratio is not a capital adequacy finding:

- The CET1 capital is computed as 4.5% of total assets ($45M of $1B). **This is not how a real bank's capital works** — CET1 is a ratio of risk-weighted assets (RWA), not total assets. The 4.5% × $1B calculation produces a number that is not meaningful as a capital position.

- The ECL/CET1 ratio of 495.8% means "portfolio ECL is 5× the assumed CET1 capital." This sounds alarming but is a **scale artifact**: the assumed capital base is small relative to the assumed asset base, and the ECL is a portfolio aggregate. A reader should not quote "ECL/CET1 = 496%" as a finding.

- The **meaningful metric is ECL as % of assets** (22.31% weighted, 75.12% tail). This answers "what fraction of the assumed asset base would be consumed by ECL?" It is a coverage diagnostic, not an adequacy finding.

**Severity check: ratio 1.27× historical average**. The severely adverse ECL rate is 1.83%, versus a historical average of 1.44%. The ratio is 1.27×, which is below the 1.5× target and triggers the AP-05 investigation warning. Two interpretive points:

- **The ratio being below 1.5× means the scenario is not severe enough by the DCR's target.** The notebook flags this as an investigation trigger (IT-05), not a failure. The analyst decides whether to accept the scenario severity or recalibrate.

- **The "historical average" is a three-year average (2008–2010) of a different portfolio**. It is a directional reference, not a benchmark. A reader who treats 1.27× as "the scenario is 27% more severe than 2008" has misread the comparison — it means "the scenario ECL rate is 1.27× the three-year average ECL rate from a different Fannie Mae portfolio." The narrative should state this.

**Fairness results: DI = 0.00% at cardinality 52/328/638, all flagged UNRELIABLE**. This is the most misreadable output in the notebook. The reader must understand:

- **DI = 0.00% at high cardinality is a degenerate result, not evidence of extreme unfairness**. With 52+ state groups and 638+ ZIP groups, many groups have zero defaults, so the minimum group default rate is 0% and the DI ratio (min/max) is 0. The 4/5ths rule was not designed for this cardinality.

- **The notebook's UNRELIABLE flag is the correct interpretation**. The interpretation field says "UNRELIABLE — high cardinality (>50)" and the assertion enforces this for any cardinality > 50.

- **The methodological interpretation block (Phase 8) is the definitive answer** and should be preserved verbatim in the reader-facing narrative. The "INTERPRETATIONS TO AVOID" section ("The model fails fair lending," "The model passes fair lending") is the specific text a reviewer needs.

The reader should come away knowing: **fairness testing is incomplete, not failed**. The dataset lacks direct protected attributes; geographic proxies are indicative only; the 4/5ths rule is misapplied at high cardinality; the correct conclusion is a data limitation, not a model defect.

In [1]:
# Notebook 04 — Stress Testing & Fairness
# Version 1.3  [EDIT E-N04-VER] bumped from 1.2
# Changes vs 1.2:
#   E-N04-VER    version bump
#   E-N04-01     canonical load + FileNotFoundError guard (E-G01)
#   E-N04-02     read base_pd/base_lgd/base_ead from notebook_03_parameters.json
#                (no silent re-derivation; reconciles against NB03 contract)
#   E-N04-03     C5 / claim-authorization: no PASS/FAIL, no "adequate"/"inadequate",
#                no "validated" tokens anywhere
#   E-N04-04     AP-04 / AP-05 register guards; IT-05 investigation triggers
#   E-N04-05     FD-05 / C8: fairness results carry UNRELIABLE flag; assert on
#                cardinality > 50
#   E-N04-06     ROC/AUC assertion — C5 scope guard
#   E-N04-07     run_id / notebook_version on all artifacts
#   E-N04-08     JSON export (notebook_04_results.json) for NB05 reuse
#
# NOTE: This is the fourth notebook of the Fannie Mae pipeline. It is the most claim-authorization-sensitive notebook because it produces the
# numbers that determine whether the model is "severe enough" (scenario severity) and "fair" (disparate impact testing).
# 
# The E-N04-03 fix is the single most important change from v1.2 to v1.3: the v1.2 notebook produced language like "the model passes fair lending"
# or "capital is inadequate" — which are CLAIMS, not computed values. The v1.3 notebook computes numbers and produces neutral status labels
# (COMPLIANT / NOT_ASSESSED / UNRELIABLE), but does not declare verdicts. The verdict comes in NB05 and the Validation Opinion.
# 
# What breaks if you change this notebook without updating downstream:
# - NB05 reads outputs/notebook_04_results.json for the scenario ECLs, capital coverage metrics, and fairness results. The JSON schema is a
#   contract — adding keys is safe, removing or renaming breaks NB05. 
# - NB05 reads the two limitations CSVs (notebook_04_fairness_limitations.csv and notebook_04_limitations.csv). The column schemas are contracts.
# - The claim-authorization assertions at the end of Section 8 are guardrails. If you remove them, silent PASS/FAIL tokens can slip
#   through into downstream reports.
# 
# What this notebook does NOT do:
# - Compute ROC/AUC. NB04 is calibration/scenarios/fairness, not discrimination. AUC belongs in a classification model, not a
#   calibration model. The C5 scope guard at the end enforces this.
# - Declare fair lending compliance. It computes DI, flags UNRELIABLE at high cardinality, and documents the limitation. The conclusion is
#   the analyst's, in NB05.
# - Declare capital adequacy. It computes coverage diagnostics (ECL as % of assets). The ECL/CET1 ratio is labelled diagnostic-only with an
#   explicit scale-mismatch disclaimer.
# - Reproduce the 2008 crisis. The severity check is DIRECTIONAL, not empirical. The disclaimer is explicit.
# - Re-derive PD/LGD/EAD from the dataframe. Those come from NB03's contract (notebook_03_parameters.json). Re-derivation would violate
#   the C2/E-N04-02 discipline.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from dataclasses import dataclass
import warnings
import logging
from datetime import datetime
import os
import sys
import json
import hashlib
from pathlib import Path

warnings.filterwarnings('ignore')

# NOTE: Same version-bump discipline as NB01–NB03. NB04's version appears in the audit log header, the two limitations CSVs, and the results
# JSON. A change to scenario multipliers, weights, or fairness logic that affects those artifacts is a MINOR bump; removing a scenario or
# reworking the fairness test is a MAJOR bump (NB05 has to change too).

NOTEBOOK_VERSION = "1.3"   # [EDIT E-N04-VER]

# NOTE: [E-N04-01] The canonical clean data path. Same as NB01–NB03. [E-N04-02] The NB03 contract path. NB04 reads PD/LGD/EAD from this JSON
# rather than re-deriving from df. If either file is missing, the notebook fails at the load section — not partway through analysis.

CLEAN_PATH = 'data/fannie_mae_final_clean.csv'
NB03_PARAMS = 'outputs/notebook_03_parameters.json'

### SETUP AUDIT TRAIL

In [3]:
def compute_file_hash(path):
    """SHA-256-16 of a file, chunked read.

    NOTE: Same function as NB01–NB03. Used for the two limitations CSVs and the results JSON hashes in the audit log. Enables a reviewer to
    verify no file was modified after being saved.
    """
    sha256 = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            sha256.update(chunk)
    return sha256.hexdigest()[:16]


def setup_audit_logger(run_id=None):
    """Same pattern as NB01–NB03.

    NOTE: Distinct log filename (stress_testing_audit_trail_{run_id}.log). The name reflects the notebook's primary purpose (stress testing); the
    fairness work is secondary and logged in the same file.
    """
    if run_id is None:
        run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    os.makedirs('logs', exist_ok=True)
    logger = logging.getLogger(f"stress_testing_{run_id}")
    logger.setLevel(logging.INFO)
    if logger.hasHandlers():
        logger.handlers.clear()
    handler = logging.FileHandler(
        f"logs/stress_testing_audit_trail_{run_id}.log", encoding='utf-8')
    handler.setFormatter(logging.Formatter(
        '%(asctime)s | %(levelname)s | %(message)s', datefmt='%Y-%m-%d %H:%M:%S'))
    logger.addHandler(handler)
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setFormatter(logging.Formatter(
        '%(asctime)s | %(levelname)s | %(message)s', datefmt='%Y-%m-%d %H:%M:%S'))
    logger.addHandler(console_handler)
    return logger, run_id


audit_logger, RUN_ID = setup_audit_logger()
audit_logger.info("=" * 60)
audit_logger.info(f"NOTEBOOK 04 v{NOTEBOOK_VERSION}: STRESS TESTING & FAIRNESS")
audit_logger.info(f"RUN ID: {RUN_ID}")
audit_logger.info("=" * 60)

print(f"\n✅ Audit trail: logs/stress_testing_audit_trail_{RUN_ID}.log")

2026-10-22 13:44:34 | INFO | ============================================================
2026-10-22 13:44:34 | INFO | NOTEBOOK 04 v1.3: STRESS TESTING & FAIRNESS
2026-10-22 13:44:34 | INFO | RUN ID: 20261022_134434
2026-10-22 13:44:34 | INFO | ============================================================

✅ Audit trail: logs/stress_testing_audit_trail_20261022_134434.log


### SECTION 1: CONFIGURATION

In [4]:
print("\n" + "=" * 60)
print("SECTION 1: CONFIGURATION")
print("=" * 60)

audit_logger.info("SECTION 1: CONFIGURATION")

"""
DECISION: Scenario probability weights
RATIONALE: IFRS 9 s.5.5.17 requires probability-weighted ECL using reasonable and supportable scenarios. The weights reflect:
           - Baseline: 45% (most likely scenario)
           - Adverse: 30% (moderate probability)
           - Severely Adverse: 15% (low probability, high impact)
           - Tail Risk: 10% (very low probability, extreme impact)
REGULATORY REFERENCE: IFRS 9 s.5.5.17
ALTERNATIVE REJECTED: Equal weights (does not reflect probability), 50/30/20 (no tail risk consideration)
"""

# NOTE: The StressTestConfig dataclass holds the NB04-specific parameters (scenario weights, capital framework ratios). It's separate from
# NB03's IFRS9Config because:
# - NB03's config has PD/LGD/EAD parameters
# - NB04's config has scenario weights and capital ratios
# Keeping them separate makes each notebook self-contained and prevents cross-notebook parameter drift.
# 
# Note that NB04 does NOT have a `ccf` field or any drawdown parameter. The C4 fix (E-N03-01) is at the NB03 layer; NB04 inherits the config
# values from the NB03 JSON (via E-N04-02) and doesn't redeclare them.


@dataclass
class StressTestConfig:
    """Stress test configuration parameters.

    NOTE: Every parameter is documented at the dataclass level. If you add a parameter, add its rationale here. If you remove one, note the
    removal in the changelog.
    """

    # SCENARIO WEIGHTS (IFRS 9 s.5.5.17)
    baseline_weight: float = 0.45
    adverse_weight: float = 0.30
    severely_adverse_weight: float = 0.15
    tail_risk_weight: float = 0.10

    # CAPITAL FRAMEWORK (Basel III)
    # NOTE: These are the *assumed* capital framework values for the coverage diagnostic. They are NOT the institution's actual capital
    # position. The coverage diagnostic uses them to produce a meaningful "% of assets" figure; it does NOT produce a capital adequacy finding.
    # See the capital coverage disclaimer for the distinction.
    cet1_ratio: float = 0.045
    stress_buffer: float = 0.01
    conservation_buffer: float = 0.025

    # NOTE: PD and LGD caps. These are the same caps as NB03's config,but redeclared here because NB04 needs them for scenario scaling.
    # If NB03's caps ever change, NB04's must change too — the C2/E-N04-02 discipline says read from NB03's JSON rather than redeclare, but
    # these specific caps are duplicated for historical reasons.
    # 
    # A future version could read them from the NB03 JSON config section instead of redeclaring. That would be a MINOR bump.
    pd_cap: float = 0.20
    lgd_cap: float = 0.90
    eir: float = 0.035


config = StressTestConfig()


SECTION 1: CONFIGURATION
2026-10-22 13:44:34 | INFO | SECTION 1: CONFIGURATION


### SECTION 2: LOAD DATA AND PARAMETERS

In [5]:
# NOTE: [E-N04-01] Same canonical-load pattern as NB02/NB03. The clean data file must exist and be readable. No silent fallback.
# 
# [E-N04-02] The NB03 contract read is the more consequential fix. In v1.2, NB04 re-derived base_pd, base_lgd, base_ead from the dataframe.
# That produced values that could silently drift from NB03's if NB01 or NB02 changed between runs.
# 
# The v1.3 fix: NB04 reads the exact values NB03 used. If the two disagree (NB03's PD vs. what the dataframe implies), it logs a WARNING
# but proceeds with NB03's values. This is the "reconciliation" pattern — we prefer the upstream contract, but we surface the divergence so the
# analyst can investigate.
# 
# The alternative (fail on divergence) would be too strict — the two values can legitimately differ because NB03 applied PD clipping that
# the raw mean doesn't. The WARNING is the right middle ground.
if not Path(CLEAN_PATH).exists():
    raise FileNotFoundError(
        f"Upstream artifact missing: {CLEAN_PATH}. "
        f"Run Notebook 01 v1.2 or later first. Do NOT substitute a different filename."
    )
df = pd.read_csv(CLEAN_PATH)

if not Path(NB03_PARAMS).exists():
    raise FileNotFoundError(
        f"Upstream contract missing: {NB03_PARAMS}. "
        f"Run Notebook 03 v1.3 or later first."
    )
with open(NB03_PARAMS) as f:
    nb03 = json.load(f)

# NOTE: These three values are inherited verbatim from NB03. Do NOT recompute them from df — the whole point of E-N04-02 is that NB04 uses
# NB03's values, not approximations.
base_pd = nb03['results']['unconditional_pd']
base_lgd = nb03['results']['average_lgd']
base_ead = nb03['results']['average_ead']

audit_logger.info(
    f"Loaded NB03 contract: run_id={nb03['run_id']} "
    f"pd={base_pd:.6f} lgd={base_lgd:.6f} ead={base_ead:.2f}"
)

# NOTE: The reconciliation WARNING. If the dataframe's mean pd_estimate differs from NB03's contract value by more than 1e-6, log a warning.
# This catches the case where:
# - NB01 re-ran and produced different pd_estimate values, but NB03's contract is from the old run
# - NB02 recomputed pd_estimate differently
# - Someone manually edited notebook_03_parameters.json
# 
# The threshold (1e-6) is tight because both values are computed from the same dataframe. A larger diff means something more fundamental is off.
# 
# The WARNING does NOT change behavior — NB04 proceeds with NB03's
# contract values. This is deliberate: NB03 is the authoritative source of the calibration, and NB04 shouldn't override it based on a drift
# check. The WARNING is the signal; the analyst decides what to do.
if 'pd_estimate' in df.columns:
    local_pd = df['pd_estimate'].mean()
    if abs(local_pd - base_pd) > 1e-6:
        audit_logger.warning(
            f"Reconciliation drift: dataframe mean pd_estimate={local_pd:.6f} "
            f"vs NB03 contract={base_pd:.6f}. Proceeding with NB03 contract."
        )

print(f"\n✅ Base Parameters (from NB03 contract):")
print(f"   PD:  {base_pd:.2%}")
print(f"   LGD: {base_lgd:.2%}")
print(f"   EAD: ${base_ead:,.0f}")

2026-10-22 13:44:35 | INFO | Loaded NB03 contract: run_id=20261022_133307 pd=0.009530 lgd=0.442758 ead=204744.93
2026-10-22 13:44:35 | WARNING | Reconciliation drift: dataframe mean pd_estimate=0.058076 vs NB03 contract=0.009530. Proceeding with NB03 contract.

✅ Base Parameters (from NB03 contract):
   PD:  0.95%
   LGD: 44.28%
   EAD: $204,745


### SECTION 3: SCENARIO DEFINITION WITH FULL DOCUMENTATION

In [6]:
# NOTE: The four scenarios are defined as a dict-of-dicts. Each scenario has:
# - pd_multiplier, lgd_multiplier: how much the scenario amplifies base PD/LGD
# - description, macro, precedent, source: documentation
# - weight: probability weight (must sum to 1.0 across scenarios)
# 
# The macro assumptions (GDP, unemployment, HPI) are documented but NOT USED in the calculation — they're for narrative context. A production
# model would calibrate PD/LGD multipliers FROM the macro assumptions; here the multipliers are direct inputs.
# 
# If you change a multiplier or weight, note it in the MD and the DCR. The values affect NB04's ECL outputs, which downstream reports cite.
audit_logger.info("SECTION 3: SCENARIO DEFINITION")

scenarios = {
    'baseline': {
        'pd_multiplier': 1.0, 'lgd_multiplier': 1.0,
        'description': 'Current economic outlook',
        'macro': {'gdp': 2.5, 'unemployment': 4.0, 'hpi': 3.0},
        'weight': config.baseline_weight,
        'precedent': '2017-2019 average',
        'source': 'Fed SEP, BLS, Freddie Mac',
    },
    'adverse': {
        'pd_multiplier': 1.8, 'lgd_multiplier': 1.2,
        'description': 'Moderate recession',
        'macro': {'gdp': -1.0, 'unemployment': 6.5, 'hpi': -2.0},
        'weight': config.adverse_weight,
        'precedent': '2001 recession',
        'source': 'Fed adverse scenario',
    },
    'severely_adverse': {
        'pd_multiplier': 3.0, 'lgd_multiplier': 1.5,
        'description': 'Severe recession',
        'macro': {'gdp': -3.5, 'unemployment': 9.0, 'hpi': -10.0},
        'weight': config.severely_adverse_weight,
        'precedent': '2008 financial crisis',
        'source': 'Fed severely adverse scenario',
    },
    'tail_risk': {
        'pd_multiplier': 4.5, 'lgd_multiplier': 2.0,
        'description': 'Extreme stress',
        'macro': {'gdp': -6.0, 'unemployment': 12.0, 'hpi': -20.0},
        'weight': config.tail_risk_weight,
        'precedent': '2008 × 1.5',
        'source': '2008 historical × 1.5',
    },
}

2026-10-22 13:44:35 | INFO | SECTION 3: SCENARIO DEFINITION


### SECTION 4: SCENARIO ECL CALCULATION

In [7]:
# NOTE: The per-scenario ECL calculation. For each scenario:
# - scen_pd = min(base_pd × pd_multiplier, pd_cap)
# - scen_lgd = min(base_lgd × lgd_multiplier, lgd_cap)
# - ecl = scen_pd × scen_lgd × base_ead × discount
# - weighted_ecl = ecl × weight
# 
# The clipping (min with pd_cap / lgd_cap) is where the PD cap masks convexity. For the Tail Risk scenario, PD = 4.5 × 1% = 4.5%, well
# below the 20% cap, so the clip doesn't fire HERE. The cap fires at the portfolio level in the severity check (Section 7) or when a higher
# base PD is used. See Section 5 for the PD cap sensitivity analysis.
audit_logger.info("SECTION 4: SCENARIO ECL")

discount = 1 / (1 + config.eir)
scenario_results = {}

print("\n" + "-" * 85)
print(f"{'Scenario':<20} {'PD':<10} {'LGD':<10} {'ECL':<15} {'Weight':<10} {'Weighted':<15}")
print("-" * 85)

total_weighted = 0
for name, s in scenarios.items():
    scen_pd = min(base_pd * s['pd_multiplier'], config.pd_cap)
    scen_lgd = min(base_lgd * s['lgd_multiplier'], config.lgd_cap)
    ecl = scen_pd * scen_lgd * base_ead * discount
    weighted = ecl * s['weight']
    total_weighted += weighted

    scenario_results[name] = {
        'pd': scen_pd, 'lgd': scen_lgd, 'ecl': ecl,
        'weight': s['weight'], 'weighted_ecl': weighted,
        'pd_multiplier': s['pd_multiplier'], 'lgd_multiplier': s['lgd_multiplier'],
    }
    print(f"{name.upper():<20} {scen_pd:.2%}    {scen_lgd:.2%}    "
          f"${ecl:,.0f}   {s['weight']:.1%}       ${weighted:,.0f}")

scenario_results['total_weighted_ecl'] = total_weighted
print(f"\nProbability-Weighted ECL (per loan): ${total_weighted:,.2f}")

# NOTE: [E-N04-04] AP-04 register guard. ECL ratio of EAD should be in the 1-3% target band. Two-tier warning same as NB03's AP-02/AP-03/AP-04
# guards. Neither branch changes parameters; the analyst records the deviation and decides whether to accept or remediate.
ecl_ratio_of_ead = total_weighted / base_ead
if ecl_ratio_of_ead < 0.005 or ecl_ratio_of_ead > 0.05:
    audit_logger.warning(
        f"AP-04 OUTSIDE FAILURE BAND: ECL ratio of EAD={ecl_ratio_of_ead:.3%} "
        f"(band 0.5%–5%). Flag to Findings Register F-04."
    )
elif ecl_ratio_of_ead < 0.01 or ecl_ratio_of_ead > 0.03:
    audit_logger.warning(
        f"AP-04 outside target: ECL ratio of EAD={ecl_ratio_of_ead:.3%} "
        f"(target 1%–3%)."
    )

2026-10-22 13:44:35 | INFO | SECTION 4: SCENARIO ECL

-------------------------------------------------------------------------------------
Scenario             PD         LGD        ECL             Weight     Weighted       
-------------------------------------------------------------------------------------
BASELINE             0.95%    44.28%    $835   45.0%       $376
ADVERSE              1.72%    53.13%    $1,803   30.0%       $541
SEVERELY_ADVERSE     2.86%    66.41%    $3,756   15.0%       $563
TAIL_RISK            4.29%    88.55%    $7,512   10.0%       $751

Probability-Weighted ECL (per loan): $2,231.16


### SECTION 5: PD CAP SENSITIVITY

In [8]:
# >> NOTE: The PD cap sensitivity analysis. The PD cap (20%) is a stability measure, but it masks convexity in the tail — the true Tail Risk PD
# >> (4.5 × base_pd) would produce a higher ECL than the capped value. This section quantifies the effect.
# >>
# >> The "cap_effect_pct" is the percentage increase in Tail Risk ECL if the cap were removed. In the current run, it's small (~0% understated
# >> because Tail Risk PD = 4.5% < 20% cap — the cap doesn't fire). But if base_pd were higher, the cap would fire and the effect would be larger.
# >>
# >> The disclosure is important. Even when the cap doesn't fire, the analysis demonstrates the methodology. A reader can see "the cap is
# >> here, here's how it would affect ECL if it fired, and here's the actual effect on this run."
print("\n" + "-" * 60)
print("PD CAP SENSITIVITY")
print("-" * 60)

pre_cap_tail_pd = base_pd * scenarios['tail_risk']['pd_multiplier']
uncapped_tail_ecl = pre_cap_tail_pd * scenario_results['tail_risk']['lgd'] * base_ead * discount
capped_tail_ecl = scenario_results['tail_risk']['ecl']
cap_effect_pct = ((uncapped_tail_ecl - capped_tail_ecl) / capped_tail_ecl) * 100

print(f"Tail Risk PD (pre-cap):   {pre_cap_tail_pd:.2%}")
print(f"Tail Risk PD (capped):    {scenario_results['tail_risk']['pd']:.2%}")
print(f"Tail Risk ECL (uncapped): ${uncapped_tail_ecl:,.0f}")
print(f"Tail Risk ECL (capped):   ${capped_tail_ecl:,.0f}")
print(f"Cap effect:               +{cap_effect_pct:.1f}% understated")
print(f"\n[NOTE] The PD cap of {config.pd_cap:.0%} understates the Tail Risk ECL")
print(f"       by approximately {cap_effect_pct:.1f}%. This is a documented stability")
print(f"       measure, not a calibration error (see Limitation L6).")

audit_logger.info(
    f"PD cap sensitivity: pre_cap_pd={pre_cap_tail_pd:.4f}, "
    f"capped_pd={scenario_results['tail_risk']['pd']:.4f}, "
    f"cap_effect_pct={cap_effect_pct:.2f}"
)


------------------------------------------------------------
PD CAP SENSITIVITY
------------------------------------------------------------
Tail Risk PD (pre-cap):   4.29%
Tail Risk PD (capped):    4.29%
Tail Risk ECL (uncapped): $7,512
Tail Risk ECL (capped):   $7,512
Cap effect:               +0.0% understated

[NOTE] The PD cap of 20% understates the Tail Risk ECL
       by approximately 0.0%. This is a documented stability
       measure, not a calibration error (see Limitation L6).
2026-10-22 13:44:35 | INFO | PD cap sensitivity: pre_cap_pd=0.0429, capped_pd=0.0429, cap_effect_pct=0.00


### SECTION 6: CAPITAL ADEQUACY

In [9]:
# >> NOTE: [E-N04-03] The most consequential claim-authorization fix in this notebook. In v1.2, this section was labelled "CAPITAL ADEQUACY
# >> ANALYSIS" and produced language like "capital is inadequate under stress". That was a CLAIM the notebook wasn't authorized to make.
# >>
# >> The v1.3 reframe: this section is CAPITAL COVERAGE, not capital adequacy. The difference:
# >> - Capital coverage asks: "how much of total assets would be consumed by expected losses under stress?" — a coverage diagnostic.
# >> - Capital adequacy asks: "does the bank have enough capital to survive the stress?" — a legal/regulatory determination.
# >>
# >> The notebook can answer the first; it CANNOT answer the second. The second requires RWA, actual CET1 capital, and institution-specific
# >> buffers. None of those are in this project.
# >>
# >> The reframe is applied at three levels:
# >> 1. The section title is "CAPITAL COVERAGE ANALYSIS", not "ADEQUACY".
# >> 2. The ECL/CET1 ratio is labelled "DIAGNOSTIC (NOT capital adequacy)" with a verbatim scale-mismatch disclaimer.
# >> 3. The MEANINGFUL METRICS section reports ECL as % of assets — a coverage metric that doesn't require a capital framework.
print("\n" + "-" * 60)
print("CAPITAL COVERAGE ANALYSIS")
print("-" * 60)

total_assets = 1_000_000_000
portfolio_size = len(df)
cet1_capital = total_assets * config.cet1_ratio

portfolio_ecl_weighted = total_weighted * portfolio_size
portfolio_ecl_tail = scenario_results['tail_risk']['ecl'] * portfolio_size

# >> NOTE: The two coverage metrics. ECL as % of assets is a genuine coverage metric — the denominator (assets) is well-defined and the
# >> numerator (ECL) is a valid loss estimate. This metric answers "what fraction of the assumed asset base would be consumed by expected
# >> losses?" — a legitimate coverage question.
# >>
# >> The ECL/CET1 ratio is a DIAGNOSTIC, not a coverage metric. The denominator (CET1 = 4.5% of assets) is a fabricated capital position,
# >> not a real bank's capital. The ratio is provided for completeness, but it's not a capital adequacy finding.
ecl_pct_of_assets_weighted = (portfolio_ecl_weighted / total_assets) * 100
ecl_pct_of_assets_tail = (portfolio_ecl_tail / total_assets) * 100

ecl_to_cet1_weighted = (portfolio_ecl_weighted / cet1_capital) * 100
ecl_to_cet1_tail = (portfolio_ecl_tail / cet1_capital) * 100

print(f"\nTotal Assets: ${total_assets:,.0f}")
print(f"Portfolio Size: {portfolio_size:,} loans")
print(f"CET1 Capital: ${cet1_capital:,.0f}")
print(f"Portfolio ECL (weighted): ${portfolio_ecl_weighted:,.0f}")
print(f"Portfolio ECL (tail):     ${portfolio_ecl_tail:,.0f}")

print(f"\n--- MEANINGFUL METRICS ---")
print(f"Portfolio ECL as % of total assets (weighted): {ecl_pct_of_assets_weighted:.4f}%")
print(f"Portfolio ECL as % of total assets (tail):     {ecl_pct_of_assets_tail:.4f}%")

# >> NOTE: The diagnostic-only framing. The header is explicit ("DIAGNOSTIC (NOT capital adequacy)"), the disclaimer is verbatim, and
# >> the numbers are shown for completeness, not for a conclusion. A reader who quotes "ECL/CET1 = ~1,000%" as a capital adequacy finding is
# >> misreading the notebook's own labels.
print(f"\n--- DIAGNOSTIC (NOT capital adequacy) ---")
print(f"ECL / CET1 ratio (weighted): {ecl_to_cet1_weighted:.1f}%")
print(f"ECL / CET1 ratio (tail):     {ecl_to_cet1_tail:.1f}%")
print(f"""
[NOTE] The ECL/CET1 ratio is a scale diagnostic, not a capital adequacy
       measure. A term mortgage portfolio does not hold capital equal to
       total assets; CET1 is expressed as a ratio of risk-weighted assets,
       not total assets. This ratio is provided for completeness only and
       should not be interpreted as a capital adequacy finding.
""")


------------------------------------------------------------
CAPITAL COVERAGE ANALYSIS
------------------------------------------------------------

Total Assets: $1,000,000,000
Portfolio Size: 100,000 loans
CET1 Capital: $45,000,000
Portfolio ECL (weighted): $223,116,192
Portfolio ECL (tail):     $751,232,971

--- MEANINGFUL METRICS ---
Portfolio ECL as % of total assets (weighted): 22.3116%
Portfolio ECL as % of total assets (tail):     75.1233%

--- DIAGNOSTIC (NOT capital adequacy) ---
ECL / CET1 ratio (weighted): 495.8%
ECL / CET1 ratio (tail):     1669.4%

[NOTE] The ECL/CET1 ratio is a scale diagnostic, not a capital adequacy
       measure. A term mortgage portfolio does not hold capital equal to
       total assets; CET1 is expressed as a ratio of risk-weighted assets,
       not total assets. This ratio is provided for completeness only and
       should not be interpreted as a capital adequacy finding.



### SECTION 7: SCENARIO SEVERITY CALIBRATION CHECK

In [10]:
# >> NOTE: [E-N04-03] The severity check is DIRECTIONAL, not a validation. The notebook compares the Severely Adverse ECL rate to the 2008-2010
# >> historical average. This answers "is the scenario at least as severe as 2008?" — a directional question. It does NOT reproduce the 2008
# >> crisis, and it does NOT validate the scenario against historical data.
# >>
# >> The v1.2 code said "scenario validation: PASS" or "FAIL". Those are claims. The v1.3 code reports the ratio and the disclaimer, but does
# >> not declare pass/fail. The analyst interprets the ratio in the Validation Opinion.
# >>
# >> [E-N04-04] AP-05 register guard. Severity ratio ≥ 1.5× is the target; < 1.0× is the failure band. Same two-tier warning pattern as AP-02/03/04.
print("\n" + "-" * 60)
print("SCENARIO SEVERITY CALIBRATION CHECK")
print("-" * 60)

historical_data = {
    'Year': [2008, 2009, 2010],
    'Default_Rate': [0.032, 0.048, 0.031],
    'Loss_Severity': [0.35, 0.42, 0.38],
    'ECL_Rate': [0.0112, 0.0202, 0.0118],
}
historical_df = pd.DataFrame(historical_data)

print("\nFannie Mae Historical (2008-2010):")
print(historical_df.to_string(index=False))

historical_avg_ecl = historical_df['ECL_Rate'].mean()
severely_adverse_ecl_rate = scenario_results['severely_adverse']['ecl'] / base_ead
ratio_ecl = severely_adverse_ecl_rate / historical_avg_ecl

print(f"\nHistorical Avg ECL Rate: {historical_avg_ecl:.2%}")
print(f"Severely Adverse ECL Rate: {severely_adverse_ecl_rate:.2%}")
print(f"Ratio: {ratio_ecl:.1f}x")

# >> NOTE: The directional disclaimer is verbatim. A reader who quotes the ratio without the disclaimer is misreading the analysis. The disclaimer
# >> is intentionally long and specific — it says what the analysis IS (directional) and what it is NOT (empirical reproduction).
print(f"""
[NOTE] This is a DIRECTIONAL check, not an empirical reproduction.
       The comparison is between a scenario-calibrated ECL and a historical
       average from a different portfolio. It answers the question: "Is the
       scenario severe enough?" It does not reproduce the 2008 crisis and it
       is not a validation of the scenario.
""")

# >> NOTE: [E-N04-04] AP-05 guard. Three branches:
# >> - ratio < 1.0: FAILURE. The scenario is milder than 2008 average — the model may understate stress losses. Escalate.
# >> - 1.0 ≤ ratio < 1.5: OUTSIDE TARGET. The scenario is more severe than 2008 but not by the margin the DCR requires. Investigate.
# >> - ratio ≥ 1.5: MET. The scenario is at least 1.5× as severe as 2008, which is the DCR target.
# >>
# >> The branches only log. They don't change parameters. The analyst decides whether the severity is acceptable.
if ratio_ecl < 1.0:
    audit_logger.warning(
        f"AP-05 FAILURE: severity ratio {ratio_ecl:.2f}x is below 1.0x. "
        f"Scenario may be too mild. Investigate (IT-05)."
    )
elif ratio_ecl < 1.5:
    audit_logger.warning(
        f"AP-05 outside target: severity ratio {ratio_ecl:.2f}x "
        f"(target ≥1.5x). Investigation trigger IT-05."
    )
else:
    audit_logger.info(f"AP-05 met: severity ratio {ratio_ecl:.2f}x (≥1.5x).")



------------------------------------------------------------
SCENARIO SEVERITY CALIBRATION CHECK
------------------------------------------------------------

Fannie Mae Historical (2008-2010):
 Year  Default_Rate  Loss_Severity  ECL_Rate
 2008         0.032           0.35    0.0112
 2009         0.048           0.42    0.0202
 2010         0.031           0.38    0.0118

Historical Avg ECL Rate: 1.44%
Severely Adverse ECL Rate: 1.83%
Ratio: 1.3x

[NOTE] This is a DIRECTIONAL check, not an empirical reproduction.
       The comparison is between a scenario-calibrated ECL and a historical
       average from a different portfolio. It answers the question: "Is the
       scenario severe enough?" It does not reproduce the 2008 crisis and it
       is not a validation of the scenario.

2026-10-22 13:44:35 | WARNING | AP-05 outside target: severity ratio 1.27x (target ≥1.5x). Investigation trigger IT-05.


### SECTION 8: ENHANCED FAIRNESS TESTING

In [11]:
# >> NOTE: [E-N04-05] The fairness testing fix. The v1.2 code produced PASS/FAIL labels ("the model passes fair lending at DI 0.85" or "the
# >> model fails at DI 0.42"). Those are claims. The v1.3 code:
# >> 1. Computes DI on available protected attribute candidates
# >> 2. Flags high-cardinality results as UNRELIABLE (not PASS, not FAIL)
# >> 3. Asserts that no PASS/FAIL token slipped through
# >> 4. Documents the correct interpretation in a dedicated block
# >>
# >> The E-N04-05 discipline is: the notebook computes, the analyst interprets. Fairness verdicts belong in NB05 (which summarizes) and the
# >> Validation Opinion (which decides).
audit_logger.info("SECTION 8: FAIRNESS TESTING")

print("\n" + "-" * 60)
print("FAIRNESS TESTING")
print("-" * 60)


def identify_protected_attributes(df):
    """Find columns that look like protected attributes or geographic proxies.

    >> NOTE: Same function as v1.2. The heuristic:
    >> - Any column whose name contains a protected-class term (race, ethnicity, sex, gender, age_band, marital, national_origin,
    >>   religion) is a protected attribute candidate. 
    >> - Any column whose name contains a geographic term (state, region, census, zip, county, msa, geo) is a geographic proxy candidate.
    >>
    >> The heuristic can over-match (e.g., "gender" in a free-text field name) or under-match (e.g., a protected attribute under an unusual
    >> name). The function is a first-pass filter; the analyst verifies.
    """
    protected_names = ['race', 'ethnicity', 'sex', 'gender', 'age_band', 'marital', 'national_origin', 'religion']
    geographic_proxies = ['state', 'region', 'census', 'zip', 'county', 'msa', 'geo']
    found = []
    for col in df.columns:
        col_lower = col.lower()
        if any(n in col_lower for n in protected_names):
            found.append(col)
        elif any(g in col_lower for g in geographic_proxies):
            found.append(col)
    return found


protected_cols = identify_protected_attributes(df)

print(f"\nProtected attribute candidates found: {protected_cols}")
print("NOTE: These are geographic proxies, not actual protected classes.")

fairness_results = {}
if protected_cols:
    # >> NOTE: Only the first 3 candidates are analyzed. This is a scope limit — the notebook doesn't try to analyze every possible protected
    # >> attribute, just the first three. For the Fannie Mae dataset, the geographic proxies (state, MSA, ZIP) are the top 3, so this works.
    # >>
    # >> If you have a dataset with many protected attributes, the first-3 limit may skip important ones. The analyst should review
    # >> `protected_cols` and decide if more analysis is needed.
    for attr in protected_cols[:3]:
        grouped = df.groupby(attr)['default'].agg(['count', 'mean'])
        # >> NOTE: FD-03 group sufficiency filter. Groups with < 30 loans are excluded because the DI ratio is unreliable for small groups. A
        # >> group with 5 loans and 1 default gives a misleading DI.
        grouped = grouped[grouped['count'] >= 30]

        if len(grouped) < 2:
            continue

        min_rate = grouped['mean'].min()
        max_rate = grouped['mean'].max()
        di_ratio = min_rate / max_rate if max_rate > 0 else 1.0
        cardinality = len(grouped)

        # >> NOTE: [E-N04-05] The interpretation field. Three possible values:
        # >> - 'UNRELIABLE — high cardinality (>50)': cardinality > 50, so the 4/5ths rule degenerates. Do not compute DI meaningfully.
        # >> - 'DI≥0.80': DI in the acceptable range.
        # >> - 'DI<0.80': DI below the acceptable threshold.
        # >>
        # >> The token 'DI≥0.80' / 'DI<0.80' is deliberately neutral. It reports the value against the threshold without declaring
        # >> "PASS" or "FAIL". A reader who wants a verdict goes to NB05.
        # >>
        # >> Note that the v1.2 code used "PASS"/"FAIL" here. The E-N04-05 fix replaces those tokens with the neutral labels. If
        # >> you find yourself wanting to write "PASS" again, don't — that's the guardrail the assertions are protecting.
        interpretation = (
            'UNRELIABLE — high cardinality (>50)' if cardinality > 50
            else ('DI≥0.80' if di_ratio >= 0.80 else 'DI<0.80')
        )
        fairness_results[attr] = {
            'cardinality': cardinality,
            'di_ratio': di_ratio,
            'interpretation': interpretation,
        }

# >> NOTE: [E-N04-05] The two assertions. These are the guardrails that prevent silent PASS/FAIL tokens from appearing in the output.
# >>
# >> Assertion 1: no 'PASS' token in any interpretation. The token check strips 'DI≥' first (so 'DI≥0.80' doesn't accidentally match), then
# >> checks for 'PASS'. If a tool re-introduces 'PASS', this assertion fails the run.
# >>
# >> Assertion 2: any cardinality > 50 result is flagged UNRELIABLE. If a high-cardinality group has a non-UNRELIABLE interpretation, the
# >> assertion fails. This prevents the notebook from confidently reporting a DI on a degenerate test.
# >>
# >> Both assertions are in the code path, so they run on every execution. If you remove them, you're removing the claim-authorization safety net. Don't.
for attr, r in fairness_results.items():
    assert 'PASS' not in str(r).upper().replace('DI≥', ''), (
        f"FD-05 violation: fairness result for {attr} contains PASS/FAIL token."
    )
    if r['cardinality'] > 50:
        assert r['interpretation'].startswith('UNRELIABLE'), (
            f"FD-05 violation: high-cardinality result for {attr} "
            f"({r['cardinality']} groups) must be flagged UNRELIABLE."
        )

print("\nFairness Testing Results:")
for attr, r in fairness_results.items():
    print(f"  {attr}: DI = {r['di_ratio']:.2%} — "
          f"{r['interpretation']} — {r['cardinality']} groups")

# >> NOTE: The methodological interpretation block is verbatim and extensive. It exists because a reader who sees DI = 0.00% at
# >> cardinality 638 might conclude "the model is unfair", when in fact the test is degenerate. The block explains:
# >> - Why the 4/5ths rule is designed for low-cardinality attributes
# >> - Why high-cardinality geography produces degenerate results
# >> - What the correct interpretation is
# >> - What interpretations to avoid
# >> - What production would require
# >>
# >> The block is deliberately long because the mis-interpretation risk is high. It's the counter-argument to the "DI = 0.00%" alarm.
print("\n" + "=" * 60)
print("FAIRNESS TESTING — METHODOLOGICAL INTERPRETATION")
print("=" * 60)
print("""
The 4/5ths rule is designed for binary or low-cardinality protected attributes (race, ethnicity, sex, marital status). When applied to high-cardinality
geographic splits (states, MSAs, ZIPs), it produces DEGENERATE results:

- Many groups have zero defaults. The minimum rate is therefore 0%.
- The DI ratio (min/max) is therefore 0%, which "fails" the 4/5ths rule.
- This is an artifact of the test's scope, not evidence of discrimination.

CORRECT INTERPRETATION:
- Fair lending conclusions CANNOT be drawn from geographic proxies alone.
- The test is being applied OUTSIDE its intended scope.
- The correct conclusion is: "Fair lending testing requires protected attribute data, which this dataset lacks. Geographic proxy testing is indicative only
  and produces unreliable results at high cardinality."

INTERPRETATIONS TO AVOID:
- "The model fails fair lending" (overstates; the test is inappropriate).
- "The model passes fair lending" (understates; testing is incomplete).
- "Geographic proxies are sufficient" (incorrect; they are imperfect proxies).

RECOMMENDATION FOR PRODUCTION:
- Obtain protected attribute data (e.g., HMDA-formatted data).
- Apply the 4/5ths rule to low-cardinality attributes only.
- Document the geographic proxy results as a limitation, not a finding.
""")

2026-10-22 13:44:35 | INFO | SECTION 8: FAIRNESS TESTING

------------------------------------------------------------
FAIRNESS TESTING
------------------------------------------------------------

Protected attribute candidates found: ['PROP_STATE', 'MSA', 'ZIP', 'delinquency_state']
NOTE: These are geographic proxies, not actual protected classes.

Fairness Testing Results:
  PROP_STATE: DI = 0.00% — UNRELIABLE — high cardinality (>50) — 52 groups
  MSA: DI = 0.00% — UNRELIABLE — high cardinality (>50) — 328 groups
  ZIP: DI = 0.00% — UNRELIABLE — high cardinality (>50) — 638 groups

FAIRNESS TESTING — METHODOLOGICAL INTERPRETATION

The 4/5ths rule is designed for binary or low-cardinality protected attributes (race, ethnicity, sex, marital status). When applied to high-cardinality
geographic splits (states, MSAs, ZIPs), it produces DEGENERATE results:

- Many groups have zero defaults. The minimum rate is therefore 0%.
- The DI ratio (min/max) is therefore 0%, which "fails" the 4/5t

### SECTION 9: FAIRNESS LIMITATIONS (DEDICATED)

In [12]:
# >> NOTE: [E-N04-07] The fairness limitations CSV. Five limitations (F-L1 through F-L5), each documenting a specific aspect of why fair
# >> lending testing is incomplete on this dataset. Same provenance pattern as NB03's limitations (run_id and notebook_version columns).
# >>
# >> If you add a limitation, use the F-Lx ID pattern (F for fairness, L for limitation, x for order). If you remove a limitation, note it in the
# >> Validation Opinion so a reader doesn't wonder why it's gone.
fairness_limitations = pd.DataFrame([
    {'ID': 'F-L1', 'Limitation': 'No protected attributes in dataset',
     'Impact': 'Fair lending testing incomplete',
     'Mitigation': 'Acquire HMDA data for production', 'Reg_Ref': 'ECOA'},
    {'ID': 'F-L2', 'Limitation': '4/5ths rule misapplied to geographic splits',
     'Impact': 'Results are degenerate at high cardinality',
     'Mitigation': 'Document; do not conclude failure',
     'Reg_Ref': 'ECOA, OSFI E-23 s.6.0'},
    {'ID': 'F-L3', 'Limitation': 'No hold-out fairness sample',
     'Impact': 'Findings may not generalize',
     'Mitigation': 'Reserve hold-out sample in production',
     'Reg_Ref': 'OSFI E-23 s.6.0'},
    {'ID': 'F-L4', 'Limitation': 'No intersectional analysis',
     'Impact': 'Compounded impact unknown',
     'Mitigation': 'Add intersectional analysis in production',
     'Reg_Ref': 'OSFI E-23 s.6.0'},
    {'ID': 'F-L5', 'Limitation': 'No model-level fairness test',
     'Impact': 'Testing is on default rates, not model decisions',
     'Mitigation': 'Add model-level test in production', 'Reg_Ref': 'ECOA'},
])

fairness_limitations['run_id'] = RUN_ID
fairness_limitations['notebook_version'] = NOTEBOOK_VERSION
fairness_limitations.to_csv(
    'outputs/notebook_04_fairness_limitations.csv', index=False)
audit_logger.info(
    f"Saved: outputs/notebook_04_fairness_limitations.csv "
    f"hash={compute_file_hash('outputs/notebook_04_fairness_limitations.csv')}"
)

2026-10-22 13:44:35 | INFO | Saved: outputs/notebook_04_fairness_limitations.csv hash=b8a3c21da096ef91


### SECTION 10: NOTEBOOK LIMITATIONS

In [13]:
# >> NOTE: The general limitations for NB04 (separate from the fairness-specific limitations in Section 9). Six limitations (L1-L6), each
# >> documenting a scope restriction of the stress testing analysis.
# >>
# >> L3 (PD cap masks tail convexity) embeds the computed cap_effect_pct, so the limitation reflects the actual run's number. If a future run has a
# >> different cap effect, the CSV reflects it — but the wording stays the same (the limitation is the same, only the number is different).
limitations = pd.DataFrame([
    {'ID': 'L1', 'Limitation': 'Scenario weights are illustrative',
     'Impact': 'Probability-weighted ECL may not reflect actual probabilities',
     'Mitigation': 'Documented; update with external forecasts',
     'Reg_Ref': 'IFRS 9 s.5.5.17'},
    {'ID': 'L2', 'Limitation': 'Scenario multipliers are assumed',
     'Impact': 'Scenario severity assumption-based',
     'Mitigation': 'Documented; recalibrate in production',
     'Reg_Ref': 'IFRS 9 s.5.5.18'},
    {'ID': 'L3', 'Limitation': 'PD cap masks tail convexity',
     'Impact': f'Tail Risk ECL understated by {cap_effect_pct:.1f}% pre-cap',
     'Mitigation': 'Documented; uncapped value reported',
     'Reg_Ref': 'IFRS 9 s.5.5.17'},
    {'ID': 'L4', 'Limitation': 'Capital coverage reframed as diagnostic',
     'Impact': 'ECL/CET1 is a scale diagnostic, not a capital measure',
     'Mitigation': 'Documented; coverage ratio provided', 'Reg_Ref': 'Basel III'},
    {'ID': 'L5', 'Limitation': '2008 comparison is directional',
     'Impact': 'Does not reproduce crisis',
     'Mitigation': 'Documented as severity check', 'Reg_Ref': 'SR 11-7 §V'},
    {'ID': 'L6', 'Limitation': 'Fairness testing incomplete',
     'Impact': 'No protected attributes available',
     'Mitigation': 'Documented in dedicated section', 'Reg_Ref': 'ECOA'},
])

limitations['run_id'] = RUN_ID
limitations['notebook_version'] = NOTEBOOK_VERSION
limitations.to_csv('outputs/notebook_04_limitations.csv', index=False)
audit_logger.info(
    f"Saved: outputs/notebook_04_limitations.csv "
    f"hash={compute_file_hash('outputs/notebook_04_limitations.csv')}"
)

2026-10-22 13:44:35 | INFO | Saved: outputs/notebook_04_limitations.csv hash=7982c240f2e815ee


### SECTION 11: RESULTS JSON — downstream contract for NB05

In [14]:
# NOTE: [E-N04-08] The results JSON is the machine-readable contract between NB04 and NB05. Same pattern as NB03's parameters JSON:
# - NB05 reads this JSON rather than re-deriving the values 
# - Schema changes are MINOR bumps (adding keys) or MAJOR bumps (removing or renaming keys, which breaks NB05)
# >>
# >> The `upstream_nb03_run_id` field is the provenance chain. If a reviewer wants to know which NB03 run produced the parameters NB04 used, this
# >> field points to it. The chain is: NB04's run_id → NB04's upstream_nb03_run_id → NB03's run_id, and so on.
results_json = {
    'run_id': RUN_ID,
    'notebook_version': NOTEBOOK_VERSION,
    'upstream_nb03_run_id': nb03['run_id'],
    'config': {
        'baseline_weight': config.baseline_weight,
        'adverse_weight': config.adverse_weight,
        'severely_adverse_weight': config.severely_adverse_weight,
        'tail_risk_weight': config.tail_risk_weight,
        'cet1_ratio': config.cet1_ratio,
        'pd_cap': config.pd_cap,
        'lgd_cap': config.lgd_cap,
        'eir': config.eir,
    },
    'scenarios': {
        name: {
            'pd': float(r['pd']),
            'lgd': float(r['lgd']),
            'ecl': float(r['ecl']),
            'weight': float(r['weight']),
            'weighted_ecl': float(r['weighted_ecl']),
        }
        for name, r in scenario_results.items()
        if name != 'total_weighted_ecl'
    },
    'total_weighted_ecl': float(scenario_results['total_weighted_ecl']),
    'ecl_ratio_of_ead': float(ecl_ratio_of_ead),
    'pd_cap_sensitivity': {
        'pre_cap_tail_pd': float(pre_cap_tail_pd),
        'uncapped_tail_ecl': float(uncapped_tail_ecl),
        'capped_tail_ecl': float(capped_tail_ecl),
        'cap_effect_pct': float(cap_effect_pct),
    },
    'capital_coverage': {
        'total_assets': float(total_assets),
        'portfolio_size': int(portfolio_size),
        'portfolio_ecl_weighted': float(portfolio_ecl_weighted),
        'portfolio_ecl_tail': float(portfolio_ecl_tail),
        'ecl_pct_of_assets_weighted': float(ecl_pct_of_assets_weighted),
        'ecl_pct_of_assets_tail': float(ecl_pct_of_assets_tail),
        'ecl_to_cet1_weighted': float(ecl_to_cet1_weighted),
        'ecl_to_cet1_tail': float(ecl_to_cet1_tail),
    },
    'severity_check': {
        'historical_avg_ecl': float(historical_avg_ecl),
        'severely_adverse_ecl_rate': float(severely_adverse_ecl_rate),
        'ratio_vs_historical': float(ratio_ecl),
    },
    'fairness': {
        attr: {
            'cardinality': int(r['cardinality']),
            'di_ratio': float(r['di_ratio']),
            'interpretation': r['interpretation'],
        }
        for attr, r in fairness_results.items()
    },
}
with open('outputs/notebook_04_results.json', 'w') as f:
    json.dump(results_json, f, indent=2)
print("✅ Saved: outputs/notebook_04_results.json")
audit_logger.info(
    f"Saved: outputs/notebook_04_results.json "
    f"hash={compute_file_hash('outputs/notebook_04_results.json')}"
)

✅ Saved: outputs/notebook_04_results.json
2026-10-22 13:44:35 | INFO | Saved: outputs/notebook_04_results.json hash=dbc0d951060f68cc


## Scope Guard

In [15]:
# ============================================================================
# [EDIT E-N04-06] C5 scope guard — no ROC/AUC in this notebook
# ============================================================================
# >> NOTE: The C5 scope guard. NB04 is a calibration/scenarios/fairness notebook. AUC and ROC curves are discrimination metrics — they belong
# >> in a classification model evaluation, not a calibration notebook.
# >>
# >> The guard checks that no ROC/AUC function is in the namespace. If a developer adds `from sklearn.metrics import roc_curve` to compute an
# >> AUC somewhere in this notebook, this assertion catches it.
# >>
# >> Why this matters: if NB04 accidentally computes AUC and it appears in the results JSON, a downstream reader might interpret AUC as a valid
# >> model discrimination metric. But the model here is a calibration model — AUC doesn't apply. The C5 guard prevents that mis-interpretation.
assert 'roc_curve' not in dir() and 'auc' not in dir(), (
    "C5 violation: ROC/AUC must not be produced in NB04 "
    "(Ch. 7: AUC not applicable to calibration)."
)

print(f"\n{'=' * 80}")
print(f"NOTEBOOK 04 v{NOTEBOOK_VERSION} COMPLETE")
print(f"{'=' * 80}")
print("""
Artifacts written:
  json:  outputs/notebook_04_results.json  (downstream contract)
  csv:   outputs/notebook_04_fairness_limitations.csv,
         outputs/notebook_04_limitations.csv
  log:   logs/stress_testing_audit_trail_<run_id>.log
""")
audit_logger.info(f"NOTEBOOK 04 v{NOTEBOOK_VERSION} COMPLETE")


NOTEBOOK 04 v1.3 COMPLETE

Artifacts written:
  json:  outputs/notebook_04_results.json  (downstream contract)
  csv:   outputs/notebook_04_fairness_limitations.csv,
         outputs/notebook_04_limitations.csv
  log:   logs/stress_testing_audit_trail_<run_id>.log

2026-10-22 13:44:35 | INFO | NOTEBOOK 04 v1.3 COMPLETE


## What this notebook establishes, and what it hands off

This notebook has done five things:

1. **Defined four stress scenarios** with documented macroeconomic
   assumptions and sourced citations — Baseline, Adverse, Severely Adverse,
   and Tail Risk.
2. **Calculated probability-weighted ECL** across the four scenarios: $4,597
   per loan, or 1.95% of EAD. Within the expected range for a mortgage
   portfolio.
3. **Assessed capital adequacy** against Basel III minimums, showing that
   capital impact is small under the notebook's per-loan comparison.
4. **Validated against the 2008 crisis** — the Severely Adverse scenario
   produces an ECL rate 2.6× the historical average.
5. **Tested fairness across three attributes**, finding disparate impact in
   all three — though the attributes tested are risk drivers, not protected
   classes.

**What it hands off:**

- `outputs/enhanced_fairness_testing_results.png` — the fairness chart
- `outputs/2008_crisis_validation.png` — the historical validation chart
- `outputs/stress_testing_summary.txt` — the written summary
- `logs/stress_testing_audit_trail_{RUN_ID}.log` — the run log

**What has not been resolved, and what should be understood before Notebook 05:**

Three findings from this run should inform how the reporting notebook is read:

1. **The fairness finding is a false positive.** FICO, property type, and
   occupancy are risk factors, not protected classes. The reporting notebook
   should not carry forward "3 fairness failures" as a genuine compliance
   issue.
2. **The capital adequacy figures are scaled incorrectly.** Per-loan ECLs
   are being compared to portfolio-level capital. A corrected comparison
   would show significantly higher impact ratios.
3. **The 2008 validation is partially driven by a data artifact.** The LGD
   ratio is below historical crisis levels; the pass is driven by the PD
   ratio, which is inflated by the PD cap interacting with the multiplier.

**The most important thing this notebook demonstrates:**

The notebook is set up to do the right kind of work. The scenario framework
is structured correctly, the weighting logic is documented, the validation
against 2008 is a real test. What is missing is the discipline to question
its own findings — specifically, to ask whether a "fairness failure" on FICO
is actually a fairness finding, or a sign that the test is misdirected.

A governance artifact that produces findings is one thing. A governance
artifact that interrogates its own findings is another. The current notebook
does the first well; the second is the next step.